# YummyAnime recommendations: saved model switching
Choose 1k / 5k / 10k. This notebook never fits models, rebuilds datasets, or reads evaluation/test interactions. Private histories stay only in the current kernel. Clear outputs before saving.

In [ ]:
import os
from IPython.display import display
from anime_recommender.exploration import project_root
from anime_recommender.inference.profile_session import ProfileSession
from anime_recommender.inference.model_versions import version_franchise_coverage
from anime_recommender.recommendation_exploration import profile_display

root = project_root()
if "profile_session" not in globals():
    profile_session = ProfileSession(root)

## Parameters
The approved default is 10k; 1k and 5k remain selectable. Edit model_version or top_k, then rerun the load and Top-N cells. Keep the same nickname. The session preserves fetched history and title cache; rerunning these cells never refetches the profile. Use refresh_profile=True explicitly in a separate call only if you want fresh source data.

In [ ]:
nickname = (
    os.getenv("YUMMYANIME_TEST_NICKNAME", "").strip()
    or globals().get("nickname", "")
    or input("Enter your YummyAnime nickname: ").strip()
    or "First_Light"
)

top_k = 15
show_history = False
model_version = "10k"  # "1k", "5k", "10k"

## Load the selected saved model
Model, item mapping, configuration and metadata are checksum/fingerprint-verified together. Missing or incompatible artifacts cause an explicit error. No silent version fallback. The session lazily caches each workspace. Local franchise mapping/rules remain shared and discovery never runs here.

In [ ]:
workspace = profile_session.workspace(model_version)
display(version_franchise_coverage(workspace, model_version))

## Fetch and recommend

Rerun after changing nickname. For a changed top_k, rerun only the Top-N cell using the fetched history. At most three public API calls: profile, lists (including user ratings and MAL IDs), favorites. No separate rating-statistics request is needed. Failure stops inference rather than using a partial history.

Binary positives: completed with no rating (0/missing), rating>=5 or favorite. Planned/dropped veto positivity. Watching alone is not positive; an already rated/favorite watching item follows the same existing policy. No new weights. MAL mapping identifies warm positives for fixed-item ALS fold-in. All known mapped items are excluded; empty warm context uses Popularity fallback.

Source history is cached by nickname, before mapping. Switching versions remaps this same immutable source history to the selected vocabulary, without HTTP.

In [ ]:
profile = profile_session.recommend(nickname, model_version=model_version, top_k=top_k)
view = profile_display(profile, nickname=nickname, show_history=show_history)
display(profile["requests"])
display(view["summary"].to_frame("Value"))

## Top-N

ALS scores are dot products, not probabilities. Coverage is reported both against the full local catalog and the train vocabulary. Excluded candidate items counts known train items removed from the candidate set. Unmapped items cannot be identified by dataset ID. Recommendations may be fewer than K if too few candidates remain.

Russian titles use verified local language fields or one bounded YummyAnime catalog lookup by exact MAL ID (up to 20 IDs, no pagination). Results stay in memory and are reused when changing K. Missing, ambiguous, non-Russian or failed lookups show a dash. No automatic translation. The English column prefers explicit English fields, then the original catalog title (which may be romanized). For Popularity fallback, ALS Score shows a dash; the selected model remains in the summary.

YummyAnime links are omitted: the local training catalog has no certified YummyAnime URL/ID mapping for recommended items. MAL IDs are shown without inventing YummyAnime URLs.

Franchise post-processing is configured in configs/franchise_filter.json. A small local YummyAnime viewing_order snapshot covers Attack on Titan, Demon Slayer and Cyberpunk: Edgerunners. Source data.id/index/text determines grouping, order and optional roles, not title similarity. Only the next required stage after the latest completed stage is eligible; an unavailable next stage is never skipped. Movie/TV alternatives are not separate mandatory seasons. Optional parts are deferred while an available unobserved next stage is prioritized, and remain optional rather than counted watched. No relation API requests occur online. The candidate pool expands locally up to the full train vocabulary. Exact known MAL IDs are excluded for every status. Only completed (list_id=2) proves viewing for earlier-mainline exclusion/prerequisite checks; planned/watching are never completion evidence. At most one item per known franchise is retained. Optional films/spin-offs are not automatically inferred watched, and unknown relations do not cause title-based exclusions. Scores and their relative order are preserved.

The diagnostics include known group count, covered candidate share, duplicate-season and unsuitable-continuation exclusions, final count, metadata source and partial/stale/error warnings. Missing metadata does not trigger online discovery.

A single bounded title lookup is attempted only for the first recommendation display per nickname/session. Model/K changes use local and in-memory titles only: newly recommended items without a verified Russian title show ?. Changing model_version never triggers more catalog API calls. Full vocabulary metadata coverage is reported separately from candidate coverage.

In [ ]:
workspace = profile_session.workspace(model_version)
profile = profile_session.recommend(nickname, model_version=model_version, top_k=top_k)
profile = profile_session.initial_titles(profile, nickname)
view = profile_display(profile, nickname=nickname, show_history=show_history)
display({"model_version": model_version})
display(view["recommendations"])
display(profile.get("title_error"))
display(profile.get("franchise_diagnostics"))
display(profile.get("model_franchise_coverage"))

## Optional history inspection

Set show_history=True in the parameter cell, then rerun this cell to inspect statuses, original ratings, favorite flags, positives and mapping. This does not repeat API calls or fold-in. Do not save the displayed personal data in notebook outputs.

In [ ]:
view = profile_display(profile, nickname=nickname, show_history=show_history)
display(view["history"])